In [1]:
"""
Sell Bot — Position Monitor -> IBKR Paper Trading -> MongoDB
Enhanced with trailing stops, flash crash detection, and emergency exits.
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import random
from typing import Optional
from dataclasses import dataclass

from pymongo import MongoClient
from ib_insync import *

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ─── IBKR Paper Trading ───────────────────────────────────────────────────────
IBKR_HOST = "127.0.0.1"
IBKR_PORT = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Trade Rules ──────────────────────────────────────────────────────────────
PROFIT_TARGET = 0.50           # Take profit at 50%
MAX_LOSS_PCT = 0.03            # HARD STOP: Never lose more than 3%
TRAILING_STOP_PCT = 0.05       # Give back max 5% from highs
FLASH_CRASH_PCT = 0.03         # Exit if drops 3% in single interval
LIQUIDITY_SPREAD_PCT = 0.005   # Exit if spread > 0.5%
MONITOR_INTERVAL = 30          # Check every 30 seconds
EMERGENCY_TIMEOUT = 5          # Seconds to wait for limit fill before market

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client = MongoClient("mongodb://localhost:27017/")
db = mongo_client["brkout_database"]
trades_collection = db["trades"]


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING (UTF-8 safe)
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh = logging.FileHandler("sell_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("sell_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

class IBKRSellClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()
        self._price_cache = {}  # symbol -> (price, timestamp)

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    def get_last_price(self, symbol: str) -> Optional[float]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        self.ib.sleep(3)
        self.ib.cancelMktData(contract)
        
        for label, val in [("last", ticker.last), ("ask", ticker.ask), ("bid", ticker.bid), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.info(f"{symbol}: price source='{label}' value=${float(val):.4f}")
                self._price_cache[symbol] = (float(val), datetime.now(timezone.utc))
                return float(val)
        
        log.warning(f"{symbol}: no price available from IBKR")
        return None

    def get_ticker(self, symbol: str) -> Ticker:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        return ticker

    def get_bars(self, symbol: str, n: int = 70) -> Optional[pd.DataFrame]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        
        for what_to_show in ("TRADES", "MIDPOINT"):
            raw = self.ib.reqHistoricalData(
                contract,
                endDateTime="",
                durationStr="10 D",
                barSizeSetting="5 mins",
                whatToShow=what_to_show,
                useRTH=False,
                keepUpToDate=False,
            )
            if raw:
                log.info(f"{symbol}: got {len(raw)} bars (whatToShow={what_to_show})")
                df = pd.DataFrame([
                    {"high": b.high, "low": b.low, "close": b.close, "volume": getattr(b, 'volume', 0)}
                    for b in raw
                ])
                return df.tail(n).reset_index(drop=True)
        
        log.warning(f"{symbol}: no bars returned from IBKR")
        return None

    def psar_exit_signal(self, symbol: str) -> bool:
        try:
            df = self.get_bars(symbol, n=70)
            if df is None or len(df) < 10:
                log.warning(f"{symbol}: not enough bars for PSAR")
                return False
            
            psar = self._compute_psar(df["high"].values, df["low"].values, df["close"].values)
            price = float(df["close"].iloc[-1])
            exit_signal = bool(psar[-1] > price)
            log.info(f"{symbol}: PSAR={psar[-1]:.4f} price={price:.4f} exit={exit_signal}")
            return exit_signal
        except Exception as e:
            log.error(f"{symbol}: PSAR error - {e}")
            return False

    @staticmethod
    def _compute_psar(highs, lows, closes, iaf=0.02, max_af=0.2) -> np.ndarray:
        n = len(closes)
        psar = closes.copy().astype(float)
        bull = True
        af = iaf
        hp = float(highs[0])
        lp = float(lows[0])
        
        for i in range(2, n):
            if bull:
                psar[i] = psar[i-1] + af * (hp - psar[i-1])
                psar[i] = min(psar[i], lows[i-1], lows[i-2])
                if lows[i] < psar[i]:
                    bull = False
                    psar[i] = hp
                    lp = lows[i]
                    af = iaf
                else:
                    if highs[i] > hp:
                        hp = highs[i]
                        af = min(af + iaf, max_af)
            else:
                psar[i] = psar[i-1] + af * (lp - psar[i-1])
                psar[i] = max(psar[i], highs[i-1], highs[i-2])
                if highs[i] > psar[i]:
                    bull = True
                    psar[i] = lp
                    hp = highs[i]
                    af = iaf
                else:
                    if lows[i] < lp:
                        lp = lows[i]
                        af = min(af + iaf, max_af)
        return psar

    def get_avg_fill_price_from_broker(self, symbol: str, order_id: int) -> Optional[float]:
        # Check open trades
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol 
                and trade.order.orderId == order_id 
                and trade.orderStatus.avgFillPrice > 0):
                avg = float(trade.orderStatus.avgFillPrice)
                log.info(f"{symbol}: broker avg fill (trades) = ${avg:.4f}")
                return avg

        # Check fills history
        fills = self.ib.fills()
        log.info(f"{symbol}: scanning {len(fills)} fill(s) for orderId={order_id}")
        matched = [f for f in fills if f.contract.symbol == symbol and f.execution.orderId == order_id]
        
        if matched:
            total_qty = sum(f.execution.shares for f in matched)
            total_cost = sum(f.execution.shares * f.execution.price for f in matched)
            avg = total_cost / total_qty
            log.info(f"{symbol}: fills avg fill = ${avg:.4f} (qty={total_qty})")
            return avg

        log.warning(f"{symbol}: no fill found for orderId={order_id}")
        return None

    def place_limit_sell(self, symbol: str, price: float, qty: int, emergency: bool = False) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        
        # For emergency, go 1% below bid for faster fill
        if emergency:
            price = price * 0.99
            
        order = LimitOrder("SELL", qty, round(price, 2))
        order.outsideRth = True
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        
        log.info(f"SELL LIMIT {symbol} x{qty} @ ${price:.2f} orderId={trade.order.orderId} emergency={emergency}")
        return trade

    def place_market_sell(self, symbol: str, qty: int) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        
        order = MarketOrder("SELL", qty)
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        
        log.info(f"SELL MARKET {symbol} x{qty} orderId={trade.order.orderId}")
        return trade

    def get_sell_fill_price(self, symbol: str, sell_order_id: int, fallback_price: float) -> float:
        self.ib.sleep(3)
        avg = self.get_avg_fill_price_from_broker(symbol, sell_order_id)
        if avg:
            return avg
        log.warning(f"{symbol}: sell fill not confirmed, using fallback ${fallback_price:.4f}")
        return fallback_price


# ══════════════════════════════════════════════════════════════════════════════
# RISK MANAGEMENT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def update_trailing_stop(trade: dict, current_price: float, ibkr: IBKRSellClient) -> float:
    """
    Update highest price seen and calculate trailing stop level.
    Returns the effective stop price.
    """
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    highest = trade.get("highest_price", entry_price)
    new_high = max(highest, current_price)
    
    # Calculate stops
    trailing_level = new_high * (1 - TRAILING_STOP_PCT)
    hard_stop = entry_price * (1 - MAX_LOSS_PCT)
    
    # Use whichever is higher (less risky)
    effective_stop = max(trailing_level, hard_stop)
    
    # Update DB if new high
    if new_high > highest:
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "highest_price": new_high,
                "trailing_stop": effective_stop,
                "hard_stop": hard_stop
            }}
        )
        log.info(f"{trade['symbol']}: New high ${new_high:.4f}, trailing stop @ ${effective_stop:.4f}")
    
    return effective_stop


def check_flash_crash(symbol: str, current_price: float, trade: dict) -> Optional[str]:
    """Check for sudden price drops that indicate flash crash."""
    last_price = trade.get("last_price")
    last_time = trade.get("last_price_time")
    
    # Update current price in DB for next iteration
    trades_collection.update_one(
        {"_id": trade["_id"]},
        {"$set": {
            "last_price": current_price,
            "last_price_time": datetime.now(timezone.utc)
        }}
    )
    
    if last_price and last_price > 0:
        drop_pct = (current_price - last_price) / last_price
        
        if drop_pct <= -FLASH_CRASH_PCT:
            log.warning(f"{symbol}: FLASH CRASH detected! Drop {drop_pct*100:.2f}% in {MONITOR_INTERVAL}s")
            return f"flash_crash_{abs(drop_pct)*100:.1f}pct"
    
    return None


def check_liquidity_crisis(ibkr: IBKRSellClient, symbol: str) -> bool:
    """Detect if bid-ask spread is widening (liquidity drying up)."""
    try:
        ticker = ibkr.get_ticker(symbol)
        if ticker.bid and ticker.ask and ticker.bid > 0:
            mid = (ticker.ask + ticker.bid) / 2
            spread_pct = (ticker.ask - ticker.bid) / mid
            
            if spread_pct > LIQUIDITY_SPREAD_PCT:
                log.warning(f"{symbol}: Liquidity crisis! Spread {spread_pct*100:.2f}%")
                return True
    except Exception as e:
        log.error(f"{symbol}: Error checking liquidity - {e}")
    
    return False


def calculate_atr(ibkr: IBKRSellClient, symbol: str, period: int = 14) -> Optional[float]:
    """Calculate Average True Range for volatility assessment."""
    df = ibkr.get_bars(symbol, n=period + 10)
    if df is None or len(df) < period:
        return None
    
    highs, lows, closes = df["high"], df["low"], df["close"]
    
    tr1 = highs - lows
    tr2 = abs(highs - closes.shift(1))
    tr3 = abs(lows - closes.shift(1))
    
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(period).mean().iloc[-1]
    
    return float(atr) if pd.notna(atr) else None


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_open_trades() -> list:
    return list(trades_collection.find({"status": "open"}))


def get_effective_entry_price(trade: dict, ibkr: IBKRSellClient) -> float:
    """Priority: avg_fill_price (DB) > broker lookup > entry_price (DB)"""
    if trade.get("avg_fill_price"):
        return float(trade["avg_fill_price"])

    order_id = trade.get("order_id")
    symbol = trade["symbol"]
    
    if order_id:
        broker_avg = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if broker_avg:
            trades_collection.update_one(
                {"_id": trade["_id"]},
                {"$set": {"avg_fill_price": broker_avg, "entry_price": broker_avg}}
            )
            log.info(f"{symbol}: updated DB with broker avg fill ${broker_avg:.4f}")
            return broker_avg

    log.warning(f"{symbol}: using stored entry_price=${trade['entry_price']:.4f} as fallback")
    return float(trade["entry_price"])


def close_trade(trade: dict, exit_price: float, reason: str):
    symbol = trade["symbol"]
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    pnl = (exit_price - entry_price) / entry_price

    trades_collection.update_one(
        {"_id": trade["_id"]},
        {"$set": {
            "status": "closed",
            "exit_price": exit_price,
            "exit_reason": reason,
            "exited_at": datetime.now(timezone.utc),
            "pnl_pct": round(pnl * 100, 2),
        }},
    )
    
    log.info(
        f"✓ TRADE CLOSED {symbol} | entry=${entry_price:.4f} | "
        f"exit=${exit_price:.4f} | pnl={pnl*100:.2f}% | reason={reason}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# EMERGENCY EXIT
# ══════════════════════════════════════════════════════════════════════════════

def emergency_exit(ibkr: IBKRSellClient, symbol: str, qty: int, reason: str) -> tuple[float, str]:
    """
    Emergency market exit when speed matters more than price.
    Returns (fill_price, actual_reason).
    """
    log.warning(f"{symbol}: 🚨 EMERGENCY EXIT triggered - {reason}")
    
    # Get current bid
    ticker = ibkr.get_ticker(symbol)
    current_bid = ticker.bid or ticker.last or ticker.close
    
    if not current_bid:
        log.error(f"{symbol}: No price for emergency exit!")
        return 0.0, "exit_failed_no_price"
    
    # Try aggressive limit first (1% below bid)
    aggressive_price = current_bid * 0.99
    trade = ibkr.place_limit_sell(symbol, aggressive_price, qty, emergency=True)
    order_id = trade.order.orderId
    
    # Wait briefly for fill
    for i in range(EMERGENCY_TIMEOUT):
        ibkr.ib.sleep(1)
        fill_price = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if fill_price:
            log.info(f"{symbol}: Emergency limit filled @ ${fill_price:.4f}")
            return fill_price, reason
    
    # Cancel and go market if not filled
    log.warning(f"{symbol}: Limit didn't fill in {EMERGENCY_TIMEOUT}s, switching to MARKET")
    try:
        ibkr.ib.cancelOrder(trade.order)
    except:
        pass
    
    market_trade = ibkr.place_market_sell(symbol, qty)
    ibkr.ib.sleep(2)
    
    fill_price = ibkr.get_avg_fill_price_from_broker(symbol, market_trade.order.orderId)
    if fill_price:
        return fill_price, f"{reason}_market"
    
    # Last resort: use last known price
    return current_bid * 0.98, f"{reason}_estimated"


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MONITOR LOOP
# ══════════════════════════════════════════════════════════════════════════════

def check_exit_conditions(trade: dict, current_price: float, entry_price: float, 
                         stop_level: float, ibkr: IBKRSellClient) -> Optional[str]:
    """
    Check all exit conditions in priority order.
    Returns reason string if should exit, None otherwise.
    """
    symbol = trade["symbol"]
    pnl = (current_price - entry_price) / entry_price
    
    # Priority 1: Hard stop (never lose more than MAX_LOSS_PCT)
    if current_price <= entry_price * (1 - MAX_LOSS_PCT):
        return "hard_stop_max_loss"
    
    # Priority 2: Flash crash detection
    flash_reason = check_flash_crash(symbol, current_price, trade)
    if flash_reason:
        return flash_reason
    
    # Priority 3: Liquidity crisis
    if check_liquidity_crisis(ibkr, symbol):
        return "liquidity_crisis"
    
    # Priority 4: Trailing stop (only if we've made at least 5% profit)
    if trade.get("highest_price", entry_price) >= entry_price * 1.05:
        if current_price <= stop_level:
            return "trailing_stop"
    
    # Priority 5: Profit target
    if pnl >= PROFIT_TARGET:
        return f"profit_target_{pnl*100:.1f}pct"
    
    # Priority 6: PSAR exit signal
    if ibkr.psar_exit_signal(symbol):
        return "psar_exit"
    
    # Priority 7: Original wider stop loss (backup)
    if pnl <= -0.10:  # Your original 10% stop
        return f"stop_loss_{pnl*100:.1f}pct"
    
    return None


def monitor_positions(ibkr: IBKRSellClient):
    log.info("=" * 60)
    log.info("SELL BOT STARTED - Enhanced Loss Protection")
    log.info(f"Hard Stop: {MAX_LOSS_PCT*100}% | Trailing: {TRAILING_STOP_PCT*100}% | Flash: {FLASH_CRASH_PCT*100}%")
    log.info("=" * 60)
    
    while True:
        try:
            open_trades = get_open_trades()
            log.info(f"--- Checking {len(open_trades)} open trade(s) ---")

            for trade in open_trades:
                symbol = trade["symbol"]
                qty = trade["qty"]
                
                # Get entry price
                entry_price = get_effective_entry_price(trade, ibkr)
                
                # Get current market price
                current_price = ibkr.get_last_price(symbol)
                if not current_price:
                    log.warning(f"{symbol}: No current price, skipping")
                    continue
                
                # Update trailing stop and get level
                stop_level = update_trailing_stop(trade, current_price, ibkr)
                
                # Calculate metrics
                pnl = (current_price - entry_price) / entry_price
                highest = trade.get("highest_price", entry_price)
                drawdown_from_high = (current_price - highest) / highest if highest > 0 else 0
                
                # Log status
                log.info(
                    f"{symbol}: entry=${entry_price:.4f} current=${current_price:.4f} "
                    f"pnl={pnl*100:+.2f}% high=${highest:.4f} dd={drawdown_from_high*100:.2f}% "
                    f"stop=${stop_level:.4f}"
                )
                
                # Check all exit conditions
                reason = check_exit_conditions(trade, current_price, entry_price, stop_level, ibkr)
                
                if not reason:
                    continue  # Hold position
                
                # Determine if this is an emergency
                is_emergency = any(x in reason for x in ["flash_crash", "hard_stop", "liquidity"])
                
                if is_emergency:
                    # Emergency exit with market order fallback
                    exit_price, final_reason = emergency_exit(ibkr, symbol, qty, reason)
                    close_trade(trade, exit_price, final_reason)
                else:
                    # Normal limit exit
                    sell_trade = ibkr.place_limit_sell(symbol, current_price, qty)
                    sell_order_id = sell_trade.order.orderId
                    confirmed_exit = ibkr.get_sell_fill_price(symbol, sell_order_id, current_price)
                    close_trade(trade, confirmed_exit, reason)

        except Exception as e:
            log.error(f"Monitor error: {e}", exc_info=True)

        time.sleep(MONITOR_INTERVAL)


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    log.info("=== Sell Bot Starting ===")
    ibkr = IBKRSellClient()
    
    try:
        ibkr.connect()
        monitor_positions(ibkr)
    except KeyboardInterrupt:
        log.info("Sell bot stopped by user.")
    except Exception as e:
        log.error(f"Fatal error: {e}", exc_info=True)
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()

2026-03-18 07:50:54,857 [INFO] === Sell Bot Starting ===
2026-03-18 07:50:55,014 [INFO] IBKR connected | accounts: ['DUD087866']
2026-03-18 07:50:55,016 [INFO] ============================================================
2026-03-18 07:50:55,017 [INFO] SELL BOT STARTED - Enhanced Loss Protection
2026-03-18 07:50:55,018 [INFO] Hard Stop: 3.0% | Trailing: 5.0% | Flash: 3.0%
2026-03-18 07:50:55,020 [INFO] ============================================================
2026-03-18 07:50:55,027 [INFO] --- Checking 0 open trade(s) ---
2026-03-18 07:51:25,030 [INFO] --- Checking 0 open trade(s) ---
2026-03-18 07:51:55,034 [INFO] --- Checking 0 open trade(s) ---
2026-03-18 07:52:25,038 [INFO] --- Checking 0 open trade(s) ---
2026-03-18 07:52:55,042 [INFO] --- Checking 0 open trade(s) ---
2026-03-18 07:53:25,045 [INFO] --- Checking 0 open trade(s) ---
2026-03-18 07:53:55,049 [INFO] --- Checking 0 open trade(s) ---
2026-03-18 07:54:25,052 [INFO] --- Checking 0 open trade(s) ---
2026-03-18 07:54:55,05

In [ ]:
"""
Sell Bot — Position Monitor -> IBKR Paper Trading -> MongoDB
Reads trades written by unified_bot.py from:
  database   : trading_db
  collection : trades_v2

Unified bot trade document schema (what we READ):
  instrument        str   — ticker symbol  (NOT "symbol")
  quantity          int   — share count    (NOT "qty")
  entry_price       float — fill price at entry
  highest_price     float — rolling high tracked by unified bot
  status            str   — "open" | "closed"
  entry_timestamp   datetime
  entry_trend_score float

This bot:
  1. Reads every {"status": "open"} doc from trades_v2
  2. Applies trailing stop, flash-crash, liquidity, PSAR, profit-target logic
  3. Places SELL via IBKR (limit → market fallback for emergencies)
  4. Writes exit fields back into the SAME trades_v2 document so unified_bot
     sees status="closed" and skips position management for that ticker.
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import random
from typing import Optional
from dataclasses import dataclass

from pymongo import MongoClient
from bson import ObjectId
from ib_insync import IB, Stock, LimitOrder, MarketOrder, Trade, Ticker, util

import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

PROFIT_TARGET       = 0.50    # Take profit at 50%
MAX_LOSS_PCT        = 0.03    # Hard stop — never lose more than 3%
TRAILING_STOP_PCT   = 0.05    # Give back max 5% from peak
FLASH_CRASH_PCT     = 0.03    # Exit if price drops 3% in one interval
LIQUIDITY_SPREAD_PCT= 0.005   # Exit if bid-ask spread > 0.5%
MONITOR_INTERVAL    = 30      # Seconds between full scans
EMERGENCY_TIMEOUT   = 5       # Seconds to wait for limit fill before market order

# ── MongoDB — MUST match unified_bot.py exactly ───────────────────────────────
MONGO_URI        = "mongodb://localhost:27017/"
MONGO_DB         = "breakout_db"          # unified_bot: mongo_client["trading_db"]
MONGO_COLLECTION = "trades_v2"           # unified_bot: db["trades_v2"]

mongo_client       = MongoClient(MONGO_URI)
db                 = mongo_client[MONGO_DB]
trades_collection  = db[MONGO_COLLECTION]   # same collection unified_bot writes to


# ══════════════════════════════════════════════════════════════════════════════
# FIELD NAME CONSTANTS  (unified_bot schema)
# ══════════════════════════════════════════════════════════════════════════════
# Centralised so any future schema change only needs editing here.

F_SYMBOL        = "instrument"      # unified_bot stores ticker as "instrument"
F_QTY           = "quantity"        # unified_bot stores shares as "quantity"
F_ENTRY_PRICE   = "entry_price"
F_HIGHEST_PRICE = "highest_price"
F_STATUS        = "status"
F_STATUS_OPEN   = "open"
F_STATUS_CLOSED = "closed"


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt    = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh     = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh     = logging.FileHandler("sell_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("sell_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

class IBKRSellClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── Price ──────────────────────────────────────────────────────────────────

    def get_last_price(self, symbol: str) -> Optional[float]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        self.ib.sleep(3)
        self.ib.cancelMktData(contract)
        for label, val in [("last", ticker.last), ("ask", ticker.ask),
                            ("bid", ticker.bid),  ("close", ticker.close)]:
            if val and float(val) > 0:
                log.info(f"{symbol}: price source='{label}' value=${float(val):.4f}")
                return float(val)
        log.warning(f"{symbol}: no price available from IBKR")
        return None

    def get_ticker(self, symbol: str) -> Ticker:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        return ticker

    # ── Bars ───────────────────────────────────────────────────────────────────

    def get_bars(self, symbol: str, n: int = 70) -> Optional[pd.DataFrame]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        for what in ("TRADES", "MIDPOINT"):
            raw = self.ib.reqHistoricalData(
                contract, endDateTime="", durationStr="10 D",
                barSizeSetting="5 mins", whatToShow=what,
                useRTH=False, keepUpToDate=False,
            )
            if raw:
                log.info(f"{symbol}: got {len(raw)} bars (whatToShow={what})")
                df = pd.DataFrame([
                    {"high": b.high, "low": b.low,
                     "close": b.close, "volume": getattr(b, "volume", 0)}
                    for b in raw
                ])
                return df.tail(n).reset_index(drop=True)
        log.warning(f"{symbol}: no bars returned from IBKR")
        return None

    # ── PSAR ───────────────────────────────────────────────────────────────────

    def psar_exit_signal(self, symbol: str) -> bool:
        try:
            df = self.get_bars(symbol, n=70)
            if df is None or len(df) < 10:
                log.warning(f"{symbol}: not enough bars for PSAR")
                return False
            psar  = self._compute_psar(df["high"].values, df["low"].values, df["close"].values)
            price = float(df["close"].iloc[-1])
            exit_signal = bool(psar[-1] > price)
            log.info(f"{symbol}: PSAR={psar[-1]:.4f} price={price:.4f} exit={exit_signal}")
            return exit_signal
        except Exception as e:
            log.error(f"{symbol}: PSAR error — {e}")
            return False

    @staticmethod
    def _compute_psar(highs, lows, closes, iaf=0.02, max_af=0.2) -> np.ndarray:
        n    = len(closes)
        psar = closes.copy().astype(float)
        bull = True
        af   = iaf
        hp   = float(highs[0])
        lp   = float(lows[0])
        for i in range(2, n):
            if bull:
                psar[i] = psar[i-1] + af * (hp - psar[i-1])
                psar[i] = min(psar[i], lows[i-1], lows[i-2])
                if lows[i] < psar[i]:
                    bull, psar[i], lp, af = False, hp, lows[i], iaf
                elif highs[i] > hp:
                    hp = highs[i]
                    af = min(af + iaf, max_af)
            else:
                psar[i] = psar[i-1] + af * (lp - psar[i-1])
                psar[i] = max(psar[i], highs[i-1], highs[i-2])
                if highs[i] > psar[i]:
                    bull, psar[i], hp, af = True, lp, highs[i], iaf
                elif lows[i] < lp:
                    lp = lows[i]
                    af = min(af + iaf, max_af)
        return psar

    # ── Fill lookup ────────────────────────────────────────────────────────────

    def get_avg_fill_price_from_broker(self, symbol: str, order_id: int) -> Optional[float]:
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol
                    and trade.order.orderId == order_id
                    and trade.orderStatus.avgFillPrice > 0):
                avg = float(trade.orderStatus.avgFillPrice)
                log.info(f"{symbol}: broker avg fill (trades) = ${avg:.4f}")
                return avg
        fills   = self.ib.fills()
        matched = [f for f in fills
                   if f.contract.symbol == symbol and f.execution.orderId == order_id]
        if matched:
            total_qty  = sum(f.execution.shares for f in matched)
            total_cost = sum(f.execution.shares * f.execution.price for f in matched)
            avg        = total_cost / total_qty
            log.info(f"{symbol}: fills avg fill = ${avg:.4f} (qty={total_qty})")
            return avg
        log.warning(f"{symbol}: no fill found for orderId={order_id}")
        return None

    # ── Orders ─────────────────────────────────────────────────────────────────

    def place_limit_sell(self, symbol: str, price: float, qty: int,
                         emergency: bool = False) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        if emergency:
            price = price * 0.99
        order           = LimitOrder("SELL", qty, round(price, 2))
        order.outsideRth = True
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        log.info(f"SELL LIMIT {symbol} x{qty} @ ${price:.2f} "
                 f"orderId={trade.order.orderId} emergency={emergency}")
        return trade

    def place_market_sell(self, symbol: str, qty: int) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        order = MarketOrder("SELL", qty)
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        log.info(f"SELL MARKET {symbol} x{qty} orderId={trade.order.orderId}")
        return trade

    def get_sell_fill_price(self, symbol: str, sell_order_id: int,
                            fallback_price: float) -> float:
        self.ib.sleep(3)
        avg = self.get_avg_fill_price_from_broker(symbol, sell_order_id)
        if avg:
            return avg
        log.warning(f"{symbol}: sell fill not confirmed, using fallback ${fallback_price:.4f}")
        return fallback_price


# ══════════════════════════════════════════════════════════════════════════════
# RISK MANAGEMENT
# ══════════════════════════════════════════════════════════════════════════════

def update_trailing_stop(trade: dict, current_price: float) -> float:
    """
    Update highest_price in DB and return the effective stop level.
    Uses the F_HIGHEST_PRICE field that unified_bot already tracks —
    we extend it, never reset it.
    """
    entry_price = float(trade[F_ENTRY_PRICE])
    highest     = float(trade.get(F_HIGHEST_PRICE) or entry_price)
    new_high    = max(highest, current_price)

    trailing_level = new_high * (1 - TRAILING_STOP_PCT)
    hard_stop      = entry_price * (1 - MAX_LOSS_PCT)
    effective_stop = max(trailing_level, hard_stop)

    symbol = trade[F_SYMBOL]

    if new_high > highest:
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                F_HIGHEST_PRICE:  new_high,
                "trailing_stop":  effective_stop,
                "hard_stop":      hard_stop,
                "updated_at":     datetime.now(),
            }}
        )
        log.info(f"{symbol}: new high ${new_high:.4f} → trailing stop @ ${effective_stop:.4f}")

    return effective_stop


def check_flash_crash(trade: dict, current_price: float) -> Optional[str]:
    """Returns a reason string if a flash-crash drop is detected."""
    symbol     = trade[F_SYMBOL]
    last_price = trade.get("last_price")

    # Write current price for the next iteration's comparison
    trades_collection.update_one(
        {"_id": trade["_id"]},
        {"$set": {
            "last_price":      current_price,
            "last_price_time": datetime.now(timezone.utc),
        }}
    )

    if last_price and float(last_price) > 0:
        drop_pct = (current_price - float(last_price)) / float(last_price)
        if drop_pct <= -FLASH_CRASH_PCT:
            log.warning(f"{symbol}: FLASH CRASH! Drop {drop_pct*100:.2f}% in {MONITOR_INTERVAL}s")
            return f"flash_crash_{abs(drop_pct)*100:.1f}pct"
    return None


def check_liquidity_crisis(ibkr: IBKRSellClient, symbol: str) -> bool:
    try:
        ticker = ibkr.get_ticker(symbol)
        if ticker.bid and ticker.ask and ticker.bid > 0:
            mid        = (ticker.ask + ticker.bid) / 2
            spread_pct = (ticker.ask - ticker.bid) / mid
            if spread_pct > LIQUIDITY_SPREAD_PCT:
                log.warning(f"{symbol}: liquidity crisis — spread {spread_pct*100:.2f}%")
                return True
    except Exception as e:
        log.error(f"{symbol}: liquidity check error — {e}")
    return False


def check_exit_conditions(trade: dict, current_price: float, entry_price: float,
                          stop_level: float, ibkr: IBKRSellClient) -> Optional[str]:
    """
    Check all exit conditions in priority order.
    Returns a reason string if we should exit, None to hold.
    """
    symbol = trade[F_SYMBOL]
    pnl    = (current_price - entry_price) / entry_price

    # 1 — Hard stop
    if current_price <= entry_price * (1 - MAX_LOSS_PCT):
        return "hard_stop_max_loss"

    # 2 — Flash crash
    flash_reason = check_flash_crash(trade, current_price)
    if flash_reason:
        return flash_reason

    # 3 — Liquidity crisis
    if check_liquidity_crisis(ibkr, symbol):
        return "liquidity_crisis"

    # 4 — Trailing stop (only active once we've gained at least 5%)
    highest = float(trade.get(F_HIGHEST_PRICE) or entry_price)
    if highest >= entry_price * 1.05 and current_price <= stop_level:
        return "trailing_stop"

    # 5 — Profit target
    if pnl >= PROFIT_TARGET:
        return f"profit_target_{pnl*100:.1f}pct"

    # 6 — PSAR flip
    if ibkr.psar_exit_signal(symbol):
        return "psar_exit"

    # 7 — Wider backup stop loss (10%)
    if pnl <= -0.10:
        return f"stop_loss_{pnl*100:.1f}pct"

    return None


# ══════════════════════════════════════════════════════════════════════════════
# EMERGENCY EXIT
# ══════════════════════════════════════════════════════════════════════════════

def emergency_exit(ibkr: IBKRSellClient, symbol: str, qty: int,
                   reason: str) -> tuple[float, str]:
    log.warning(f"{symbol}: EMERGENCY EXIT — {reason}")
    ticker      = ibkr.get_ticker(symbol)
    current_bid = ticker.bid or ticker.last or ticker.close

    if not current_bid:
        log.error(f"{symbol}: no price for emergency exit!")
        return 0.0, "exit_failed_no_price"

    # Try aggressive limit first
    trade    = ibkr.place_limit_sell(symbol, current_bid, qty, emergency=True)
    order_id = trade.order.orderId

    for _ in range(EMERGENCY_TIMEOUT):
        ibkr.ib.sleep(1)
        fill = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if fill:
            log.info(f"{symbol}: emergency limit filled @ ${fill:.4f}")
            return fill, reason

    # Fall through to market
    log.warning(f"{symbol}: limit not filled in {EMERGENCY_TIMEOUT}s → MARKET")
    try:
        ibkr.ib.cancelOrder(trade.order)
    except Exception:
        pass

    mkt_trade = ibkr.place_market_sell(symbol, qty)
    ibkr.ib.sleep(2)
    fill = ibkr.get_avg_fill_price_from_broker(symbol, mkt_trade.order.orderId)
    if fill:
        return fill, f"{reason}_market"

    return current_bid * 0.98, f"{reason}_estimated"


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_open_trades() -> list:
    """
    Read all open trades from trades_v2 — the same collection unified_bot writes.
    Filter uses the exact field names unified_bot stores.
    """
    return list(trades_collection.find({F_STATUS: F_STATUS_OPEN}))


def get_entry_price(trade: dict) -> float:
    """
    Unified bot always writes entry_price at order time.
    avg_fill_price may also be present if unified_bot confirmed a fill.
    Priority: avg_fill_price > entry_price.
    """
    if trade.get("avg_fill_price"):
        return float(trade["avg_fill_price"])
    return float(trade[F_ENTRY_PRICE])


def close_trade(trade: dict, exit_price: float, reason: str):
    """
    Write exit fields back into trades_v2 so unified_bot treats the
    position as closed and skips it in its own position-management loop.
    Mirrors the field names unified_bot uses when it closes trades itself:
      exit_price, exit_timestamp, reason, realized_pnl, status → "closed"
    """
    symbol      = trade[F_SYMBOL]
    entry_price = get_entry_price(trade)
    qty         = int(trade[F_QTY])
    pnl_ps      = exit_price - entry_price
    pnl_pct     = pnl_ps / entry_price
    realized    = pnl_ps * qty

    trades_collection.update_one(
        {"_id": trade["_id"]},
        {"$set": {
            F_STATUS:         F_STATUS_CLOSED,
            "exit_price":     exit_price,
            "exit_timestamp": datetime.now(),
            "reason":         reason,
            "realized_pnl":   round(realized, 4),
            "pnl_pct":        round(pnl_pct * 100, 2),
            "updated_at":     datetime.now(),
            "closed_by":      "sell_bot",       # audit trail
        }}
    )
    log.info(
        f"TRADE CLOSED | {symbol} | entry=${entry_price:.4f} | "
        f"exit=${exit_price:.4f} | pnl={pnl_pct*100:+.2f}% (${realized:+.2f}) | {reason}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MONITOR LOOP
# ══════════════════════════════════════════════════════════════════════════════

def monitor_positions(ibkr: IBKRSellClient):
    log.info("=" * 60)
    log.info("SELL BOT STARTED")
    log.info(f"DB: {MONGO_DB}.{MONGO_COLLECTION}  |  "
             f"Hard stop: {MAX_LOSS_PCT*100}%  |  "
             f"Trailing: {TRAILING_STOP_PCT*100}%  |  "
             f"Flash: {FLASH_CRASH_PCT*100}%")
    log.info("=" * 60)

    while True:
        try:
            open_trades = get_open_trades()
            log.info(f"--- {len(open_trades)} open trade(s) in {MONGO_DB}.{MONGO_COLLECTION} ---")

            for trade in open_trades:
                # ── Read using unified_bot field names ────────────────────────
                symbol = trade[F_SYMBOL]      # "instrument" field
                qty    = int(trade[F_QTY])    # "quantity" field

                entry_price = get_entry_price(trade)

                current_price = ibkr.get_last_price(symbol)
                if not current_price:
                    log.warning(f"{symbol}: no price, skipping this cycle")
                    continue

                stop_level = update_trailing_stop(trade, current_price)

                pnl            = (current_price - entry_price) / entry_price
                highest        = float(trade.get(F_HIGHEST_PRICE) or entry_price)
                drawdown       = (current_price - highest) / highest if highest > 0 else 0

                log.info(
                    f"{symbol}: entry=${entry_price:.4f} | current=${current_price:.4f} | "
                    f"pnl={pnl*100:+.2f}% | high=${highest:.4f} | "
                    f"dd={drawdown*100:.2f}% | stop=${stop_level:.4f} | qty={qty}"
                )

                # ── Re-load trade to get last_price set in previous cycle ─────
                # (check_flash_crash reads trade["last_price"] then updates it)
                trade = trades_collection.find_one({"_id": trade["_id"]})

                reason = check_exit_conditions(
                    trade, current_price, entry_price, stop_level, ibkr
                )

                if not reason:
                    continue   # hold

                is_emergency = any(x in reason for x in
                                   ["flash_crash", "hard_stop", "liquidity"])

                if is_emergency:
                    exit_price, final_reason = emergency_exit(
                        ibkr, symbol, qty, reason
                    )
                    close_trade(trade, exit_price, final_reason)
                else:
                    sell_trade  = ibkr.place_limit_sell(symbol, current_price, qty)
                    order_id    = sell_trade.order.orderId
                    exit_price  = ibkr.get_sell_fill_price(symbol, order_id, current_price)
                    close_trade(trade, exit_price, reason)

        except Exception as e:
            log.error(f"Monitor loop error: {e}", exc_info=True)

        time.sleep(MONITOR_INTERVAL)


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    log.info("=== Sell Bot Starting ===")
    ibkr = IBKRSellClient()
    try:
        ibkr.connect()
        monitor_positions(ibkr)
    except KeyboardInterrupt:
        log.info("Sell bot stopped by user.")
    except Exception as e:
        log.error(f"Fatal error: {e}", exc_info=True)
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()


2026-03-20 08:19:50,457 [INFO] === Sell Bot Starting ===
2026-03-20 08:19:50,603 [INFO] IBKR connected | accounts: ['DUD087866']
2026-03-20 08:19:50,604 [INFO] ============================================================
2026-03-20 08:19:50,604 [INFO] SELL BOT STARTED
2026-03-20 08:19:50,606 [INFO] DB: breakout_db.trades_v2  |  Hard stop: 3.0%  |  Trailing: 5.0%  |  Flash: 3.0%
2026-03-20 08:19:50,607 [INFO] ============================================================
2026-03-20 08:19:50,633 [INFO] --- 0 open trade(s) in breakout_db.trades_v2 ---
